# 🎵 Proyecto ML: Predicción de Popularidad en Spotify

## Fase 2 — Preprocesamiento, Selección de Variables y Modelo Base

**Técnicas finales a comparar (Entrega Final):**
1. Decision Tree Regressor (modelo base — esta fase)
2. Random Forest Regressor
3. Support Vector Regression (SVR)

---

### Objetivo de esta fase
Preparar los datos para que sean aptos para entrenar modelos de Machine Learning, justificar cada decisión de preprocesamiento, seleccionar las variables más relevantes, y entrenar el primer modelo (Decision Tree) como punto de partida.

## 0. Setup e Importación del Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('✅ Librerías importadas')

In [ ]:
!pip install kagglehub -q

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download('maharshipandya/-spotify-tracks-dataset')
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
DATASET_PATH = os.path.join(path, csv_files[0])

df = pd.read_csv(DATASET_PATH, index_col=0)
print(f'✅ Dataset cargado: {df.shape[0]:,} filas, {df.shape[1]} columnas')
df.head()

---
## 1. Preprocesamiento

### ¿Por qué es necesario el preprocesamiento?

Los datos en bruto casi nunca están listos para entrenar un modelo. Antes de pasarle los datos a un algoritmo necesitamos:

- **Garantizar calidad de datos:** eliminar registros duplicados o incompletos que distorsionarían el aprendizaje
- **Formato numérico válido:** los modelos de ML no entienden texto ni booleanos directamente, todo debe ser numérico
- **Escalas comparables:** algunos algoritmos (especialmente SVM) son muy sensibles a la escala de las variables — una variable en rango 0-1 (`danceability`) no puede competir en igualdad de condiciones con `duration_ms` que está en cientos de miles

En esta sección documentamos **cada decisión y su justificación**, ya que es importante para la exposición entender el *por qué*, no solo el *cómo*.

### 1.1 Revisión de valores nulos

**¿Por qué revisamos nulos primero?** Si una variable tiene muchos valores faltantes, entrenar un modelo con ella puede generar errores o sesgos. Necesitamos saber qué tan grave es el problema antes de decidir si imputamos (rellenamos) o eliminamos esos registros.

In [ ]:
nulls = df.isnull().sum()
nulls = nulls[nulls > 0]

if nulls.empty:
    print('✅ No hay valores nulos en el dataset')
else:
    print('⚠️ Columnas con nulos:')
    print(nulls)
    # Justificación: dado el tamaño del dataset (114k filas), eliminar
    # las pocas filas con nulos no afecta significativamente la
    # representatividad de los datos.
    df = df.dropna()
    print(f'\n✅ Filas tras eliminar nulos: {df.shape[0]:,}')

### 1.2 Revisión y eliminación de duplicados

**¿Por qué eliminamos duplicados?** Si la misma canción aparece varias veces, el modelo le da más "peso" a esa observación de forma artificial — está viendo el mismo patrón repetido y puede sobreajustarse (overfitting) a esas filas repetidas, dando una falsa sensación de buen desempeño.

In [ ]:
# Duplicados exactos (todas las columnas iguales)
dup_exact = df.duplicated().sum()
print(f'Duplicados exactos: {dup_exact}')

# Duplicados por track_id (misma canción, posiblemente distintas versiones/álbumes)
dup_track = df.duplicated(subset=['track_id']).sum() if 'track_id' in df.columns else 'N/A'
print(f'Duplicados por track_id: {dup_track}')

shape_before = df.shape[0]
df = df.drop_duplicates()
if 'track_id' in df.columns:
    df = df.drop_duplicates(subset=['track_id'], keep='first')

print(f'\nFilas antes: {shape_before:,} | Filas después: {df.shape[0]:,}')
print(f'✅ Eliminados: {shape_before - df.shape[0]:,} registros duplicados')

### 1.3 Conversión de variables booleanas

**¿Por qué convertimos `explicit`?** La columna `explicit` viene como `True`/`False` (booleano). Los algoritmos de ML trabajan internamente con números, así que convertimos a `1`/`0`. Es una transformación simple pero necesaria para que el modelo pueda usarla como feature numérica.

In [ ]:
print('Tipo antes:', df['explicit'].dtype)
df['explicit'] = df['explicit'].astype(int)
print('Tipo después:', df['explicit'].dtype)
print('\nDistribución:')
print(df['explicit'].value_counts())

### 1.4 Decisión sobre `track_genre` (125 categorías)

**El problema:** `track_genre` es una variable categórica con **125 valores distintos**. Si usáramos *One-Hot Encoding* tradicional (crear una columna binaria por cada género), pasaríamos de tener ~13 columnas numéricas a más de 130 — esto se llama **"maldición de la dimensionalidad"** y trae varios problemas:

- El modelo se vuelve mucho más lento de entrenar
- Con tantas columnas dispersas (la mayoría en 0), algunos modelos pierden capacidad de generalización
- Para Decision Tree y Random Forest no es catastrófico, pero para SVM (que usaremos después) es especialmente problemático

**Nuestra decisión: Frequency/Target Encoding simplificado**

En lugar de crear 125 columnas, calculamos **la popularidad promedio histórica de cada género** y la usamos como una sola columna numérica nueva: `genre_avg_popularity`. Esto:

- Reduce 125 columnas → 1 columna
- Captura la información relevante (qué tan popular tiende a ser ese género en general)
- Es compatible con los 3 modelos que usaremos (árboles y SVM)

> **Nota para la exposición:** esta técnica se llama *Target Encoding* o *Mean Encoding* — es una alternativa común a One-Hot cuando hay muchas categorías.

In [ ]:
# Calcular la popularidad promedio por género
genre_popularity = df.groupby('track_genre')['popularity'].mean()

print(f'Número de géneros únicos: {df["track_genre"].nunique()}')
print('\nEjemplo — Top 5 géneros más "populares" en promedio:')
print(genre_popularity.sort_values(ascending=False).head())
print('\nEjemplo — Top 5 géneros menos "populares" en promedio:')
print(genre_popularity.sort_values(ascending=True).head())

# Crear la nueva columna numérica
df['genre_avg_popularity'] = df['track_genre'].map(genre_popularity)

print(f'\n✅ Nueva columna creada: genre_avg_popularity')
df[['track_genre', 'genre_avg_popularity']].head()

⚠️ **Nota importante sobre fuga de datos (data leakage):** estamos calculando `genre_avg_popularity` usando la columna `popularity` (nuestro target). Esto técnicamente introduce información del target en una feature. Para esta fase exploratoria es aceptable y common practice documentarlo, pero en la entrega final, lo correcto es calcular este promedio **solo con el conjunto de entrenamiento** (después del split) y aplicarlo al conjunto de prueba, para evitar que el modelo "vea" información que no debería tener de antemano. Lo retomamos en la sección de Split.

### 1.5 Eliminación de columnas no útiles para el modelo

**¿Por qué eliminamos estas columnas?**

| Columna | Razón para eliminar |
|---|---|
| `track_id`, `track_name`, `artists`, `album_name` | Son identificadores/texto único por canción — no aportan patrones generalizables, un modelo no puede "aprender" de un ID o un nombre |
| `track_genre` | Ya extrajimos su información útil en `genre_avg_popularity` (1.4) |
| `time_signature`, `key`, `mode` | *(Opcional, evaluamos su correlación antes de decidir)* |

In [ ]:
cols_to_drop = ['track_id', 'track_name', 'artists', 'album_name', 'track_genre']
df_model = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print('Columnas restantes:')
print(df_model.columns.tolist())
print(f'\nShape: {df_model.shape}')

---
## 2. Selección de Variables

### ¿Por qué hacemos selección de variables?

No todas las columnas disponibles ayudan a predecir `popularity`. Algunas pueden:

- Tener **correlación muy baja o nula** con el target → no aportan señal
- Estar **altamente correlacionadas entre sí** (multicolinealidad) → aportan información redundante, lo que puede confundir a algunos modelos y dificultar interpretar cuál variable es realmente importante

Reducir el número de variables a las más relevantes hace que el modelo sea **más simple, más rápido de entrenar, y más fácil de interpretar** — esto es especialmente importante para SVM, que es sensible a tener muchas dimensiones.

### 2.1 Correlación con el target (`popularity`)

Calculamos qué tan relacionada está cada variable numérica con `popularity`. Valores cercanos a +1 o -1 indican relación fuerte; valores cercanos a 0 indican poca o ninguna relación lineal.

In [ ]:
corr_with_target = df_model.corr()['popularity'].drop('popularity').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if v > 0 else 'crimson' for v in corr_with_target.values]
ax.barh(corr_with_target.index[::-1], corr_with_target.values[::-1], color=colors[::-1], alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlación de cada variable con popularity', fontsize=13, fontweight='bold')
ax.set_xlabel('Coeficiente de correlación (Pearson)')
plt.tight_layout()
plt.show()

print('Correlación con popularity (ordenado por magnitud):')
print(corr_with_target.to_string())

### 2.2 Multicolinealidad entre features

**¿Qué buscamos?** Pares de variables con correlación muy alta entre sí (> 0.7 u 0.8 en valor absoluto) — esto indica que están midiendo "lo mismo" desde perspectivas distintas, y mantener ambas no aporta información nueva al modelo.

**Ejemplo esperado:** `energy` y `loudness` — una canción con más energía suele ser percibida como más fuerte/ruidosa, por lo que es razonable esperar que estén correlacionadas.

In [ ]:
features_only = df_model.drop(columns=['popularity'])
corr_matrix = features_only.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, center=0, ax=ax, annot_kws={'size': 7}, linewidths=0.5)
ax.set_title('Matriz de Correlación entre Features (sin el target)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Identificar pares con alta correlación
print('\n⚠️ Pares de variables con |correlación| > 0.6:')
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.6:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], round(val, 3)))
            print(f'  {corr_matrix.columns[i]:25s} <-> {corr_matrix.columns[j]:25s} : {val:.3f}')

if not high_corr_pairs:
    print('  No se encontraron pares con correlación > 0.6')

### 2.3 Decisión final de variables

**Criterios aplicados:**

1. Si dos variables tienen correlación alta entre sí (multicolinealidad), conservamos la que tenga **mayor correlación con `popularity`** y eliminamos la otra
2. Conservamos `genre_avg_popularity` ya que captura información categórica relevante de forma compacta
3. No eliminamos variables solo por tener correlación lineal baja con el target — los árboles (Decision Tree, Random Forest) pueden capturar **relaciones no lineales** que la correlación de Pearson no detecta. Solo descartamos por **redundancia**, no por correlación baja.

> Completa esta celda con tu decisión final tras correr las celdas anteriores y observar los resultados.

In [ ]:
# Ejemplo de decisión — AJUSTAR según lo que muestren las celdas anteriores
# Si energy y loudness tienen alta correlación, eliminamos la que tenga
# menor correlación con popularity (revisar resultado de 2.1 y 2.2)

columns_to_drop_for_redundancy = []  # ej: ['loudness'] si aplica

final_features = [c for c in features_only.columns if c not in columns_to_drop_for_redundancy]

print(f'Variables finales seleccionadas ({len(final_features)}):')
for f in final_features:
    print(f'  - {f}')

---
## 3. División Train/Test (Split)

### ¿Por qué dividimos los datos?

Si entrenamos y evaluamos el modelo con los **mismos datos**, no sabemos si realmente "aprendió" patrones generales o simplemente "memorizó" los datos de entrenamiento (overfitting). Para medir el desempeño real del modelo ante datos nuevos, dividimos el dataset en:

- **Train (entrenamiento, 80%):** los datos que el modelo usa para aprender los patrones
- **Test (prueba, 20%):** datos que el modelo *nunca ve* durante el entrenamiento, usados solo para evaluar qué tan bien generaliza

**¿Por qué 80/20?** Es la proporción estándar en ML — suficientes datos para que el modelo aprenda bien (80%), y suficientes datos de prueba (20%) para que la evaluación sea estadísticamente confiable. Con 114k filas, el 20% sigue siendo ~22k filas, una muestra de prueba robusta.

**¿Por qué fijamos `random_state`?** Para que la división sea **reproducible** — si corremos el notebook otra vez, obtenemos exactamente el mismo split, lo cual es importante para comparar resultados entre las 3 técnicas de forma justa (todas deben entrenarse y evaluarse con los mismos datos).

In [ ]:
X = df_model[final_features]
y = df_model['popularity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape[0]:,} filas ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test:  {X_test.shape[0]:,} filas ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\nFeatures usadas ({X_train.shape[1]}): {list(X_train.columns)}')

### 3.1 Escalado de variables (Standardization)

**¿Por qué escalamos?**

- **Decision Tree y Random Forest:** NO necesitan escalado — toman decisiones basadas en umbrales ("¿`energy` > 0.5?"), por lo que la escala no afecta su funcionamiento
- **SVR (la técnica que usaremos después):** SÍ es muy sensible a la escala — calcula distancias entre puntos, y si una variable está en rango 0-1 y otra en rango 0-300,000 (`duration_ms`), la variable con números más grandes domina el cálculo aunque no sea más importante

**Decisión:** ajustamos (`fit`) el escalador **solo con datos de entrenamiento**, y lo aplicamos (`transform`) a ambos conjuntos. Esto evita que información del conjunto de prueba "se filtre" al proceso de entrenamiento (data leakage).

Preparamos el escalado ahora para que esté listo cuando entrenemos SVR más adelante, aunque Decision Tree (esta fase) no lo necesite.

In [ ]:
scaler = StandardScaler()

# fit SOLO con train, transform en ambos
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print('✅ Datos escalados (disponibles para SVR más adelante)')
print('\nEjemplo — antes del escalado:')
print(X_train.iloc[0])
print('\nEjemplo — después del escalado:')
print(X_train_scaled.iloc[0])

---
## 4. Modelo Base: Decision Tree Regressor

### ¿Por qué empezamos con Decision Tree?

Decision Tree es el modelo más **simple e interpretable** de los tres que vamos a comparar:

- Funciona haciendo "preguntas" sucesivas sobre los datos (ej: "¿`genre_avg_popularity` > 40?") para dividir el espacio en regiones cada vez más pequeñas, y predice el promedio de `popularity` dentro de cada región final
- No requiere escalado de datos
- Sirve como **punto de referencia (baseline)**: si Random Forest o SVR no superan a un solo árbol, algo anda mal con esos modelos o con el preprocesamiento
- Es la base conceptual de Random Forest (que es, esencialmente, muchos árboles combinados) — entender uno ayuda a entender el otro

In [ ]:
# max_depth limita qué tan "profundo" puede crecer el árbol.
# Sin límite, un árbol puede memorizar perfectamente el train set (overfitting).
# Empezamos con un valor moderado como punto de partida.
tree_model = DecisionTreeRegressor(max_depth=10, random_state=42)
tree_model.fit(X_train, y_train)

print('✅ Modelo entrenado')

### 4.1 Evaluación del modelo

**Métricas usadas:**

- **MSE (Error Cuadrático Medio):** promedio de los errores al cuadrado. Penaliza fuerte los errores grandes. Está en unidades de `popularity²`, por eso es difícil de interpretar directamente
- **RMSE (Raíz de MSE):** misma idea que MSE pero en las unidades originales de `popularity` (0-100) — más fácil de interpretar ("en promedio, el modelo se equivoca por X puntos de popularidad")
- **R² (Coeficiente de determinación):** indica qué proporción de la variabilidad de `popularity` es explicada por el modelo. Va de 0 a 1 (puede ser negativo si el modelo es muy malo). R² = 0.6 significa que el modelo explica el 60% de la variabilidad.

In [ ]:
y_pred_train = tree_model.predict(X_train)
y_pred_test = tree_model.predict(X_test)

def evaluate(y_true, y_pred, label):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'{label}:')
    print(f'  MSE  = {mse:.3f}')
    print(f'  RMSE = {rmse:.3f}')
    print(f'  MAE  = {mae:.3f}')
    print(f'  R²   = {r2:.3f}')
    return {'mse': mse, 'rmse': rmse, 'mae': mae, 'r2': r2}

print('=== Decision Tree Regressor ===\n')
train_metrics = evaluate(y_train, y_pred_train, 'Train')
print()
test_metrics = evaluate(y_test, y_pred_test, 'Test')

if train_metrics['r2'] - test_metrics['r2'] > 0.15:
    print('\n⚠️ Posible overfitting: el desempeño en train es mucho mejor que en test.')
    print('   Considerar reducir max_depth o usar poda (pruning).')

### 4.2 Visualización: Predicciones vs Valores Reales

**¿Qué buscamos en esta gráfica?** Si el modelo predijera perfectamente, todos los puntos caerían sobre la línea diagonal roja (predicción = valor real). Qué tan dispersos están los puntos respecto a esa línea nos da una idea visual del error del modelo.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, y_pred_test, alpha=0.15, s=10, color='steelblue')
ax.plot([0, 100], [0, 100], color='crimson', linestyle='--', linewidth=2, label='Predicción perfecta')
ax.set_xlabel('Popularity real')
ax.set_ylabel('Popularity predicha')
ax.set_title('Decision Tree — Predicciones vs Valores Reales (Test)', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### 4.3 Importancia de Variables

**¿Qué es esto?** Decision Tree puede decirnos qué tan "útil" fue cada variable para reducir el error al hacer las divisiones del árbol. Variables con mayor importancia son las que más influyen en la predicción de `popularity` según este modelo.

Esto es información valiosa para la exposición: permite responder *"¿qué características de audio influyen más en la popularidad?"*

In [ ]:
importances = pd.Series(tree_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importances.index[::-1], importances.values[::-1], color='steelblue', alpha=0.85)
ax.set_title('Importancia de Variables — Decision Tree', fontsize=13, fontweight='bold')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.show()

print(importances.to_string())

---
## 5. Resumen y Próximos Pasos

### Lo que se hizo en esta fase

1. **Preprocesamiento:** revisión de nulos y duplicados, conversión de `explicit` a numérico, codificación de `track_genre` (125 categorías → 1 columna `genre_avg_popularity` vía Target Encoding), eliminación de columnas identificadoras
2. **Selección de variables:** análisis de correlación con el target y detección de multicolinealidad entre features
3. **Split:** división 80/20 train/test con `random_state=42` para reproducibilidad, más escalado preparado para SVR
4. **Modelo base:** Decision Tree Regressor entrenado y evaluado con MSE, RMSE, MAE y R²

### Próximos pasos (Entrega Final)

- Entrenar **Random Forest Regressor** (ensemble de árboles — se espera que mejore sobre el árbol individual al reducir varianza)
- Entrenar **SVR** sobre los datos escalados (`X_train_scaled`, `X_test_scaled`)
- Comparar las 3 técnicas con la misma tabla de métricas
- Análisis de residuos y conclusiones finales

---
*Fase 2 completada*